# A4.4 - Deep Learning: GANs

## Contexte

**Generative Adversarial Networks** : un générateur produit de fausses images, un discriminateur distingue les vraies des fausses. Entraînement adversarial sur le dataset SVHN.

- Générateur : bruit aléatoire → image 32×32×3
- Discriminateur : image → réel ou faux

In [55]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from sklearn.metrics import accuracy_score

In [ ]:
# Ensure Jupyter kernel can find pip-installed CUDA shared libraries
import site
from pathlib import Path
import os
cuda_lib_dirs = []
for site_pkg in site.getsitepackages():
    nvidia_dir = Path(site_pkg) / "nvidia"
    if nvidia_dir.exists():
        for lib_dir in nvidia_dir.glob("*/lib"):
            cuda_lib_dirs.append(str(lib_dir))

if cuda_lib_dirs:
    existing = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(cuda_lib_dirs + ([existing] if existing else []))

# Configuration pour éviter les erreurs CuDNN
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Croissance de la mémoire GPU activée")
    except RuntimeError as e:
        print(e)



# Vérification rapide du support GPU TensorFlow
print("TensorFlow:", tf.__version__)
print("GPU détecté(s):", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU détecté(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [57]:
# Chargement des données (adapter les chemins si nécessaire)
x_train = np.load("data/p4_2026_svhn_train.npz")["images"].transpose(3, 0, 1, 2)
x_test = np.load("data/p4_2026_svhn_test.npz")["images"].transpose(3, 0, 1, 2)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")

x_train shape: (50000, 32, 32, 3)
x_test shape: (10000, 32, 32, 3)


---
## Question 1 : Template GAN

Compléter les TODO dans le template fourni.

### Discriminateur
- Input shape : (32, 32, 3)
- Conv2D + ReLU/LeakyReLU (un ou plusieurs blocs)
- Flatten + Dense(1) sans activation (logits)

In [58]:
def create_discriminator():
    model = keras.Sequential()

    # TODO: specify input shape
    model.add(keras.Input(shape=(32, 32, 3)))

    # TODO: add a Conv2D layer and a ReLU or LeakyReLU layer
    model.add(layers.Conv2D(64, kernel_size=3, strides=2, padding="same"))
    model.add(layers.LeakyReLU(alpha=0.2))

    # TODO: optional: add more blocks of Conv2D and ReLU or LeakyReLU layers
    model.add(layers.Conv2D(128, kernel_size=3, strides=2, padding="same"))
    model.add(layers.LeakyReLU(alpha=0.2))

    # TODO: add a dense layer with a single output unit and no activation function
    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

### Générateur
- Input : vecteur de bruit de dimension 100
- Dense → BatchNorm → ReLU/LeakyReLU → Reshape (ex: 4×4×128)
- Conv2DTranspose (padding="same", use_bias=False) + BatchNorm + ReLU/LeakyReLU
- Conv2DTranspose finale avec tanh, sortie 32×32×3

In [59]:
def create_generator():
    noise_dim = 100

    model = keras.Sequential()

    # TODO: specify input shape
    model.add(keras.Input(shape=(noise_dim,)))

    # TODO: specify a Dense layer
    model.add(layers.Dense(8 * 8 * 128, use_bias=False))

    # TODO: add a batch normalization layer and a ReLU or LeakyReLU layer
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # TODO: add a reshape layer (ex: 4x4x128)
    model.add(layers.Reshape((8, 8, 128)))

    # TODO: add block(s) of Conv2DTranspose, BatchNormalization, ReLU/LeakyReLU
    # Conv2DTranspose: padding="same", use_bias=False
    model.add(layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # TODO: optional extra blocks   

    # model.add(layers.Conv2DTranspose(64, kernel_size=3, strides=2, padding="same", use_bias=False))
    # model.add(layers.BatchNormalization())
    # model.add(layers.LeakyReLU(alpha=0.2))

    # TODO: final Conv2DTranspose with tanh, output 32x32x3
    model.add(layers.Conv2DTranspose(3, kernel_size=4, strides=2, padding="same", use_bias=False, activation="tanh"))

    return model

### Boucle d'entraînement
- Loss : BinaryCrossentropy(from_logits=True)
- Optimiseur : Adam, lr = 1e-4
- Générateur : tromper le discriminateur (labels réels sur les faux)
- Discriminateur : classifier correctement réels et faux

In [60]:
def train(train_images, epochs, batch_size=128):
    noise_dim = 100

    bce_from_logits = tf.keras.losses.BinaryCrossentropy(from_logits=True)

    # TODO: create the generator and discriminator
    generator = create_generator()
    discriminator = create_discriminator()

    

    # Scale train images to [-1, 1]
    train_images = train_images.astype('float32')
    train_images = (train_images - 127.5) / 127.5
    train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(len(train_images)).batch(batch_size)

    # TODO: create optimizers for both models (ex: Adam with learning rate 1e-4)
    generator_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

    for epoch in range(epochs):
        for image_batch in train_dataset:

            # TODO: sample random noise
            current_batch_size = tf.shape(image_batch)[0]
            noise = tf.random.normal([current_batch_size, noise_dim])

            with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

                # TODO: generate images (training=True)
                generated_images = generator(noise, training=True)

                # TODO: discriminator output for real and fake (training=True)
                real_output = discriminator(image_batch, training=True)
                fake_output = discriminator(generated_images, training=True)

                # labels
                real_labels = tf.ones_like(real_output)
                fake_labels = tf.zeros_like(fake_output)

                # generator loss -> generator tries to make fake images classified as real
                gen_loss = bce_from_logits(tf.ones_like(fake_output), fake_output)

                # TODO: discriminator loss
                real_loss = bce_from_logits(real_labels, real_output)
                fake_loss = bce_from_logits(fake_labels, fake_output)
                disc_loss = real_loss + fake_loss

            # apply gradients
            gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
            gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
            generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
            discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

        print(f"Epoch {epoch + 1}/{epochs} - gen_loss: {gen_loss.numpy():.4f} - disc_loss: {disc_loss.numpy():.4f}")

    # save models
    generator.save("generator.keras")
    discriminator.save("discriminator.keras")

    return generator, discriminator

### Génération d'images

In [61]:
def generate_images(generator, num_images):
    noise_dim = 100
    # TODO: sample random noise
    noise = tf.random.normal([num_images, noise_dim])

    # TODO: generate images (training=False)
    generated_images = generator(noise, training=False)

    generated_images = (generated_images + 1.0) / 2.0  # Convert back to [0, 1]
    return generated_images.numpy()

### Évaluation du discriminateur

In [62]:
def evaluate_discriminator(discriminator, real_images, fake_images):
    assert real_images.shape == fake_images.shape

    real_images = real_images.astype(np.float32)
    if real_images.max() > 1.0:
        real_images = (real_images - 127.5) / 127.5

    fake_images = fake_images.astype(np.float32)
    if fake_images.min() >= 0.0 and fake_images.max() <= 1.0:
        fake_images = fake_images * 2.0 - 1.0

    # TODO: create labels
    real_labels = np.ones((len(real_images), 1))
    fake_labels = np.zeros((len(fake_images), 1))

    # TODO: compute predictions
    real_predictions = discriminator(real_images, training=False)
    fake_predictions = discriminator(fake_images, training=False)

    predictions = np.concatenate([real_predictions, fake_predictions], axis=0)
    labels = np.concatenate([real_labels, fake_labels], axis=0)
    predicted_labels = (predictions > 0).astype(int)

    # TODO: compute accuracy
    acc = accuracy_score(labels, predicted_labels)
    return acc

---
### Entraînement

In [63]:
# Entraînement du GAN

generator, discriminator = train(x_train, epochs=50, batch_size=64)



/home/theo/Documents/UCL/Q10/ML/A4/Notebook/venv/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
W0000 00:00:1775303096.959095   20199 conv_ops_gpu.cc:328] None of the algorithms provided by cuDNN frontend heuristics worked; trying fallback algorithms.  Conv: batch: 64
in_depths: 128
out_depths: 128
in: 16
in: 16
data_format: 1
filter: 4
filter: 4
filter: 128
dilation: 1
dilation: 1
stride: 2
stride: 2
padding: 1
padding: 1
dtype: DT_FLOAT
group_count: 1
device_identifier: "sm_6.1 with 4237426688B RAM, 6 cores, 1290500KHz clock, 3504000KHz mem clock, 1048576B L2$"
version: 3

2026-04-04 13:44:56.971760: W tensorflow/core/framework/op_kernel.cc:1855] OP_REQUIRES failed at conv_grad_input_ops.cc:390 : NOT_FOUND: No algorithm worked!  Error messages:
  Profiling failure on CUDNN engine eng1{}: UNKNOWN: CUDNN_STATUS_INTERNAL_ERROR
in external/local_xla/xla/stream_executor

NotFoundError: Exception encountered when calling Conv2DTranspose.call().

[1m{{function_node __wrapped__Conv2DBackpropInput_device_/job:localhost/replica:0/task:0/device:GPU:0}} No algorithm worked!  Error messages:
  Profiling failure on CUDNN engine eng1{}: UNKNOWN: CUDNN_STATUS_INTERNAL_ERROR
in external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc(5308): 'status'
  Profiling failure on CUDNN engine eng25{}: UNKNOWN: CUDNN_STATUS_INTERNAL_ERROR
in external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc(5308): 'status'
  Profiling failure on CUDNN engine eng0{}: UNKNOWN: <unknown cudnn status: 5003>
in external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc(5308): 'status'
  Profiling failure on CUDNN engine eng1{k2=3,k3=0}: UNKNOWN: CUDNN_STATUS_INTERNAL_ERROR
in external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc(5308): 'status' [Op:Conv2DBackpropInput][0m

Arguments received by Conv2DTranspose.call():
  • inputs=tf.Tensor(shape=(64, 8, 8, 128), dtype=float32)

### Visualisation des images générées

In [ ]:
# Visualiser quelques images générées
import matplotlib.pyplot as plt

num_images = 16
generated_images = generate_images(generator, num_images)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(np.clip(generated_images[i], 0.0, 1.0))
    ax.axis("off")
plt.tight_layout()
plt.show()

### Évaluation

In [ ]:
# Évaluer le discriminateur
num_eval = min(1000, x_test.shape[0])
fake_images = generate_images(generator, num_eval)
acc = evaluate_discriminator(discriminator, x_test[:num_eval], fake_images)
print(f"Accuracy du discriminateur (réel vs faux): {acc:.4f}")

---
## Question 2 : Upload du générateur

Uploader `generator.keras` sur INGInious.

Évalué sur sa capacité à tromper un discriminateur externe.

In [ ]:
# Sauvegarde du générateur
generator.save("generator.keras")
print("Générateur sauvegardé: generator.keras")

---
## Question 3 : Upload du discriminateur

Uploader `discriminator.keras` sur INGInious.

Évalué sur sa capacité à distinguer des vraies images de fausses images générées par un générateur externe.

In [ ]:
# Sauvegarde du discriminateur
discriminator.save("discriminator.keras")
print("Discriminateur sauvegardé: discriminator.keras")